In [30]:
import sqlite3
from datetime import datetime

In [31]:
with sqlite3.connect('my_database.db') as conn:
    cursor = conn.cursor()
    
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            age INTEGER,
            email TEXT UNIQUE
        )
    ''')
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS posts (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT NOT NULL UNIQUE,
            date TEXT,
            content TEXT NOT NULL,
            user_id INTEGER,
            FOREIGN KEY (user_id) REFERENCES users(id)
        )
    ''')
    
    # Сохраняем изменения
    conn.commit()
    print("База данных и таблица созданы успешно!")

База данных и таблица созданы успешно!


In [32]:
with sqlite3.connect('my_database.db') as conn:
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()
    print("Таблицы в базе:", tables)

Таблицы в базе: [('users',), ('sqlite_sequence',), ('posts',)]


In [33]:
def add_users_with_error_handling(users_list, db='my_database.db'):
    added_count = 0
    skipped_count = 0

    with sqlite3.connect(db) as conn:
        for user in users_list:
            try:
                cursor = conn.cursor()
                cursor.execute(
                    "INSERT INTO users (name, age, email) VALUES (?, ?, ?)",
                    user
                )
                conn.commit()  # коммитим каждую успешную вставку
                added_count += 1
            except sqlite3.IntegrityError:
                print(f"Пропускаем: {user[0]} ({user[2]}) — email уже существует")
                skipped_count += 1

    print(f"Добавлено: {added_count}, пропущено: {skipped_count}")
    return added_count, skipped_count


In [34]:
def add_post(post, db='my_database.db'):
    added_count = 0
    skipped_count = 0
    with sqlite3.connect(db) as conn:
        try:
            cursor = conn.cursor()
            
            cursor.execute(
                'INSERT INTO posts (title, content, user_id, date) VALUES (?, ?, ?, ?)',
                post
            )
            added_count += 1
            conn.commit()

        except sqlite3.IntegrityError:
            print(f"Пропускаем: {post} ({post[0]}) — title уже существует")
            skipped_count += 1

    print(f"Добавлено: {added_count}, пропущено: {skipped_count}")


In [35]:
users_to_add = [
    ('Анна', 25, 'anna@mail.com'),
    ('Иван', 30, 'ivan@mail.com'),
    ('Мария', 28, 'maria@mail.com'),
    ('Мария', 28, 'maria1@mail.com'),
]

post_to_add = ('Title_03', 'post content text ', 1, datetime.now().strftime('%Y-%m-%d:%H.%M.%S'))


add_users_with_error_handling(users_to_add)
add_post(post_to_add)

Пропускаем: Анна (anna@mail.com) — email уже существует
Пропускаем: Иван (ivan@mail.com) — email уже существует
Пропускаем: Мария (maria@mail.com) — email уже существует
Пропускаем: Мария (maria1@mail.com) — email уже существует
Добавлено: 0, пропущено: 4
Пропускаем: ('Title_03', 'post content text ', 1, '2026-04-08:19.00.19') (Title_03) — title уже существует
Добавлено: 0, пропущено: 1


In [36]:
with sqlite3.connect('my_database.db') as conn:
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM posts;")

    print("Response:", *cursor.fetchall(), sep='\n')


Response:
(1, 'Title_03', '2026-04-08:18.58.53', 'post content text ', 1)
